# Penggabungan Data Harga Daging dari Google Drive

Notebook ini membaca semua file Excel dari folder Google Drive (berisi data harga daging per provinsi),
lalu mengubah format **wide → long** dan menggabungkan semua file menjadi satu DataFrame dengan kolom:
`provinsi`, `kota`, `jenis_pasar`, `jenis_daging`, `harga`, `waktu`

## 1. Install & Import Library

In [ ]:
# Install library yang diperlukan
!pip install -q gdown google-api-python-client google-auth-httplib2 google-auth-oauthlib openpyxl

In [ ]:
import os
import re
import io
import pandas as pd
from pathlib import Path
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2 import service_account
from google.colab import auth
from google.auth import default

print('Library berhasil di-import!')

## 2. Autentikasi Google Drive

> **Jika menggunakan Google Colab**, jalankan cell di bawah ini untuk login otomatis.
>
> **Jika menggunakan Jupyter lokal**, gunakan Service Account atau OAuth — lihat komentar di cell.

In [ ]:
# =============================================
# OPSI A: Google Colab (jalankan ini)
# =============================================
try:
    auth.authenticate_user()   # Akan muncul pop-up login
    creds, _ = default()
    drive_service = build('drive', 'v3', credentials=creds)
    print('Autentikasi Colab berhasil!')
except Exception as e:
    print(f'Bukan Colab atau gagal: {e}')
    print('Coba gunakan OPSI B di bawah ini.')

In [ ]:
# =============================================
# OPSI B: Jupyter Lokal dengan Service Account
# Uncomment dan isi path ke file credentials JSON
# =============================================

# SERVICE_ACCOUNT_FILE = 'path/ke/credentials.json'  # Ganti dengan path Anda
# SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
# creds = service_account.Credentials.from_service_account_file(
#     SERVICE_ACCOUNT_FILE, scopes=SCOPES
# )
# drive_service = build('drive', 'v3', credentials=creds)
# print('Autentikasi Service Account berhasil!')

## 3. Konfigurasi: ID Folder Utama

In [ ]:
# ID folder utama di Google Drive
# Diambil dari URL: https://drive.google.com/drive/folders/1OHa8U6bDIIqQ4lMcvERAY7-iprjx97qH
MAIN_FOLDER_ID = '1OHa8U6bDIIqQ4lMcvERAY7-iprjx97qH'

# Daftar kata kunci daging yang akan difilter dari kolom 'Komoditas'
# Sesuaikan jika nama komoditas berbeda di data Anda
KATA_KUNCI_DAGING = [
    'daging sapi', 'daging ayam', 'daging kambing', 'daging babi',
    'daging kerbau', 'ayam ras', 'ayam kampung', 'sapi'
]

print(f'Folder ID: {MAIN_FOLDER_ID}')

## 4. Fungsi Helper

In [ ]:
def list_items_in_folder(folder_id):
    """Mengembalikan list file/folder di dalam folder_id."""
    results = []
    page_token = None
    while True:
        response = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields='nextPageToken, files(id, name, mimeType)',
            pageToken=page_token
        ).execute()
        results.extend(response.get('files', []))
        page_token = response.get('nextPageToken')
        if not page_token:
            break
    return results


def download_xlsx(file_id):
    """Download file xlsx dari Google Drive ke memori (BytesIO)."""
    request = drive_service.files().get_media(fileId=file_id)
    buf = io.BytesIO()
    downloader = MediaIoBaseDownload(buf, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    buf.seek(0)
    return buf


def parse_info_dari_nama_file(nama_file):
    """
    Ekstrak jenis_pasar dan kota dari nama file.
    Contoh: 'PEDAGANG BESAR_MATARAM.xlsx' → jenis_pasar='PEDAGANG BESAR', kota='MATARAM'
    """
    nama = nama_file.replace('.xlsx', '').replace('.XLSX', '')
    if '_' in nama:
        parts = nama.split('_', 1)
        jenis_pasar = parts[0].strip()
        kota = parts[1].strip()
    else:
        jenis_pasar = 'TIDAK DIKETAHUI'
        kota = nama.strip()
    return jenis_pasar, kota


def is_tanggal(col):
    """Cek apakah nama kolom merupakan format tanggal (mengandung '/')."""
    return isinstance(col, str) and re.search(r'\d{1,2}[/\-]\s*\d{1,2}[/\-]\s*\d{4}', col)


def bersihkan_tanggal(tgl_str):
    """Ubah string tanggal 'DD/ MM/ YYYY' atau 'DD/MM/YYYY' ke datetime."""
    try:
        tgl_bersih = re.sub(r'\s+', '', str(tgl_str))
        return pd.to_datetime(tgl_bersih, dayfirst=True)
    except:
        return pd.NaT


print('Fungsi helper berhasil didefinisikan!')

## 5. Fungsi Pemrosesan Utama

In [ ]:
def proses_satu_file(buf, provinsi, jenis_pasar, kota, kata_kunci_daging):
    """
    Membaca satu file xlsx, mengubah dari wide ke long format,
    dan menambahkan kolom: provinsi, kota, jenis_pasar.
    
    Struktur file:
    - Kolom pertama: nomor sheet
    - Kolom kedua: 'Komoditas (Rp)' = nama jenis daging
    - Kolom selanjutnya: tanggal-tanggal sebagai header
    """
    try:
        # Baca semua sheet
        all_sheets = pd.read_excel(buf, sheet_name=None, header=0)
    except Exception as e:
        print(f'    [ERROR] Gagal membaca file: {e}')
        return pd.DataFrame()
    
    frames = []
    
    for sheet_name, df in all_sheets.items():
        if df.empty or df.shape[1] < 3:
            continue
        
        # Cari kolom Komoditas (biasanya kolom ke-2)
        col_komoditas = None
        for col in df.columns:
            if isinstance(col, str) and 'komoditas' in col.lower():
                col_komoditas = col
                break
        
        # Fallback: kolom index 1 (kedua)
        if col_komoditas is None and df.shape[1] >= 2:
            col_komoditas = df.columns[1]
        
        if col_komoditas is None:
            continue
        
        # Identifikasi kolom tanggal
        cols_tanggal = [c for c in df.columns if is_tanggal(str(c))]
        
        # Jika tidak ada kolom tanggal — coba parse dari baris pertama
        if not cols_tanggal:
            print(f'    [SKIP] Sheet "{sheet_name}": tidak ada kolom tanggal ditemukan.')
            continue
        
        # Ambil hanya baris yang berisi kata kunci daging
        df_daging = df.copy()
        df_daging[col_komoditas] = df_daging[col_komoditas].astype(str).str.strip()
        
        if kata_kunci_daging:  # Filter jika ada kata kunci
            mask = df_daging[col_komoditas].str.lower().apply(
                lambda x: any(k.lower() in x for k in kata_kunci_daging)
            )
            df_daging = df_daging[mask]
        else:
            # Ambil semua baris kecuali NaN dan header-like
            df_daging = df_daging[df_daging[col_komoditas].notna()]
            df_daging = df_daging[df_daging[col_komoditas].str.lower() != 'komoditas (rp)']
        
        if df_daging.empty:
            continue
        
        # Subset ke kolom komoditas + tanggal
        df_subset = df_daging[[col_komoditas] + cols_tanggal].copy()
        df_subset = df_subset.rename(columns={col_komoditas: 'jenis_daging'})
        
        # Melt: wide → long
        df_long = df_subset.melt(
            id_vars='jenis_daging',
            value_vars=cols_tanggal,
            var_name='waktu_raw',
            value_name='harga'
        )
        
        # Bersihkan nilai waktu
        df_long['waktu'] = df_long['waktu_raw'].apply(bersihkan_tanggal)
        df_long = df_long.drop(columns=['waktu_raw'])
        
        # Tambahkan kolom identitas
        df_long['provinsi'] = provinsi
        df_long['kota'] = kota
        df_long['jenis_pasar'] = jenis_pasar
        
        frames.append(df_long)
    
    if frames:
        return pd.concat(frames, ignore_index=True)
    return pd.DataFrame()


print('Fungsi proses_satu_file berhasil didefinisikan!')

## 6. Loop Utama: Baca Semua File dari Google Drive

In [ ]:
MIME_FOLDER = 'application/vnd.google-apps.folder'
MIME_XLSX = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'

all_dataframes = []

print('Mulai membaca data dari Google Drive...\n')

# Ambil daftar folder provinsi
items_utama = list_items_in_folder(MAIN_FOLDER_ID)
folder_provinsi = [x for x in items_utama if x['mimeType'] == MIME_FOLDER]

print(f'Ditemukan {len(folder_provinsi)} folder provinsi:')
for fp in folder_provinsi:
    print(f'  - {fp["name"]}')
print()

for folder_prov in folder_provinsi:
    provinsi = folder_prov['name'].upper().strip()
    print(f'[PROVINSI] {provinsi}')
    
    items_prov = list_items_in_folder(folder_prov['id'])
    
    # Cek apakah ada sub-folder (kota) atau langsung file
    sub_folders = [x for x in items_prov if x['mimeType'] == MIME_FOLDER]
    files_langsung = [x for x in items_prov if x['mimeType'] == MIME_XLSX]
    
    if sub_folders:
        # Ada sub-folder (misalnya sub-folder kota)
        for sub in sub_folders:
            print(f'  [SUB-FOLDER] {sub["name"]}')
            items_sub = list_items_in_folder(sub['id'])
            files_sub = [x for x in items_sub if x['mimeType'] == MIME_XLSX]
            
            for f in files_sub:
                jenis_pasar, kota = parse_info_dari_nama_file(f['name'])
                print(f'    Membaca: {f["name"]} → Pasar: {jenis_pasar}, Kota: {kota}')
                
                buf = download_xlsx(f['id'])
                df_hasil = proses_satu_file(buf, provinsi, jenis_pasar, kota, KATA_KUNCI_DAGING)
                
                if not df_hasil.empty:
                    all_dataframes.append(df_hasil)
                    print(f'    → {len(df_hasil):,} baris ditambahkan')
    
    # File langsung di dalam folder provinsi
    for f in files_langsung:
        jenis_pasar, kota = parse_info_dari_nama_file(f['name'])
        print(f'  Membaca: {f["name"]} → Pasar: {jenis_pasar}, Kota: {kota}')
        
        buf = download_xlsx(f['id'])
        df_hasil = proses_satu_file(buf, provinsi, jenis_pasar, kota, KATA_KUNCI_DAGING)
        
        if not df_hasil.empty:
            all_dataframes.append(df_hasil)
            print(f'  → {len(df_hasil):,} baris ditambahkan')
    
    print()

print(f'\nSelesai! Total {len(all_dataframes)} file berhasil diproses.')

## 7. Gabungkan Semua DataFrame

In [ ]:
if all_dataframes:
    df_gabung = pd.concat(all_dataframes, ignore_index=True)
    
    # Urutkan kolom sesuai permintaan
    df_gabung = df_gabung[['provinsi', 'kota', 'jenis_daging', 'jenis_pasar', 'harga', 'waktu']]
    
    # Konversi tipe data
    df_gabung['harga'] = pd.to_numeric(df_gabung['harga'], errors='coerce')
    df_gabung['waktu'] = pd.to_datetime(df_gabung['waktu'], errors='coerce')
    
    # Bersihkan nilai harga 0 atau negatif (opsional — komentari jika tidak perlu)
    # df_gabung = df_gabung[df_gabung['harga'] > 0]
    
    print(f'Total baris: {len(df_gabung):,}')
    print(f'Total kolom: {df_gabung.shape[1]}')
    print(f'\nTipe data:')
    print(df_gabung.dtypes)
    print(f'\nPreview 5 baris pertama:')
    df_gabung.head()
else:
    print('Tidak ada data yang berhasil dibaca. Periksa koneksi atau struktur folder.')

## 8. Eksplorasi Data

In [ ]:
print('=== INFO UMUM ===')
df_gabung.info()
print()

print('=== RINGKASAN STATISTIK HARGA ===')
print(df_gabung['harga'].describe())
print()

print('=== DAFTAR PROVINSI ===')
print(df_gabung['provinsi'].unique())
print()

print('=== DAFTAR JENIS DAGING ===')
print(df_gabung['jenis_daging'].unique())
print()

print('=== DAFTAR JENIS PASAR ===')
print(df_gabung['jenis_pasar'].unique())
print()

print('=== RENTANG WAKTU ===')
print(f"Dari: {df_gabung['waktu'].min()}")
print(f"Sampai: {df_gabung['waktu'].max()}")

In [ ]:
# Cek missing values
print('=== MISSING VALUES ===')
print(df_gabung.isnull().sum())
print(f'\nPersentase missing:')
print((df_gabung.isnull().sum() / len(df_gabung) * 100).round(2))

In [ ]:
# Jumlah baris per provinsi
print('=== DISTRIBUSI DATA PER PROVINSI ===')
df_gabung.groupby('provinsi').size().sort_values(ascending=False)

In [ ]:
# Rata-rata harga per jenis daging per provinsi
print('=== RATA-RATA HARGA PER JENIS DAGING & PROVINSI ===')
(
    df_gabung
    .groupby(['provinsi', 'jenis_daging'])['harga']
    .mean()
    .round(0)
    .reset_index()
    .rename(columns={'harga': 'harga_rata2'})
    .sort_values(['provinsi', 'jenis_daging'])
)

## 9. Simpan Hasil

In [ ]:
# =============================================
# OPSI A: Simpan ke file lokal (CSV & Excel)
# =============================================
OUTPUT_CSV = 'data_harga_daging_gabungan.csv'
OUTPUT_XLSX = 'data_harga_daging_gabungan.xlsx'

df_gabung.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'CSV tersimpan: {OUTPUT_CSV}')

df_gabung.to_excel(OUTPUT_XLSX, index=False)
print(f'Excel tersimpan: {OUTPUT_XLSX}')

In [ ]:
# =============================================
# OPSI B: Upload hasil ke Google Drive
# =============================================
# Uncomment cell ini untuk meng-upload hasil ke Google Drive

# from googleapiclient.http import MediaFileUpload

# def upload_ke_drive(local_path, folder_id, mime_type):
#     """Upload file ke Google Drive folder tertentu."""
#     file_metadata = {'name': os.path.basename(local_path), 'parents': [folder_id]}
#     media = MediaFileUpload(local_path, mimetype=mime_type)
#     file = drive_service.files().create(
#         body=file_metadata, media_body=media, fields='id, name'
#     ).execute()
#     print(f"Upload berhasil: {file['name']} (ID: {file['id']})")

# OUTPUT_FOLDER_ID = 'ID_FOLDER_OUTPUT'  # Ganti dengan ID folder tujuan
# upload_ke_drive(OUTPUT_CSV, OUTPUT_FOLDER_ID, 'text/csv')
# upload_ke_drive(OUTPUT_XLSX, OUTPUT_FOLDER_ID, 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')

In [ ]:
# =============================================
# OPSI C: Download dari Colab ke komputer lokal
# =============================================
# Uncomment jika menggunakan Google Colab

# from google.colab import files
# files.download(OUTPUT_CSV)
# files.download(OUTPUT_XLSX)

## 10. Preview Data Final

In [ ]:
print(f'TOTAL BARIS DATA GABUNGAN: {len(df_gabung):,}')
print(f'KOLOM: {list(df_gabung.columns)}')
print()
df_gabung.head(20)